# Updated Cleaned Data

This notebook cleans and prepares the project datasets for analysis.

The goal of this notebook is to:

- Load the raw project CSV files
- Standardize column names
- Convert date and time columns into datetime format
- Create before/after rollout variables
- Clean response-time variables
- Create a long-format reasons dataset
- Clean SPD call, close code, and unit datasets
- Save cleaned CSV files for the analysis notebooks

This notebook should be run before the analysis notebooks.

In [61]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

## 1. Set File Paths

This section sets up the folders used by the project.

The raw data should be stored in the `data` folder. Cleaned files will also be saved back into the `data` folder so that the analysis notebooks can use them.

In [62]:
DATA_FOLDER = Path(".")

# Check what files Python can currently see
print("Current working directory:")
print(os.getcwd())

print("\nFiles in this folder:")
print(os.listdir(DATA_FOLDER))

Current working directory:
C:\Users\Phoenix M\DSiA Project

Files in this folder:
['.ipynb_checkpoints', '2015-2025 SPD Calls for Service.csv', '2015-2025 SPD Calls with Close Codes.csv', '2015-2025 SPD Responding Units.csv', 'cleaned_mcslc.csv', 'cleaned_mcslc_reasons_long.csv', 'cleaned_spd_calls.csv', 'cleaned_spd_close_codes.csv', 'cleaned_spd_relevant_calls.csv', 'cleaned_spd_units.csv', 'Clean_data.ipynb', 'data', 'Eugene_CAD_data_noloc - Copy', 'graph_boxplot_Minutes_Arrival_to_Engagement.png', 'graph_boxplot_Minutes_Dispatch_to_Arrival.png', 'graph_boxplot_Minutes_Request_to_Dispatch.png', 'graph_dispatch_endpoints_before_after.png', 'graph_dispatch_reason_categories_before_after.png', 'graph_dispatch_reason_percentages_before_after.png', 'graph_dispositions_before_after.png', 'graph_mean_bar_Minutes_Request_to_Dispatch.png', 'graph_mean_Minutes_Arrival_to_Departure_by_city_period.png', 'graph_mean_Minutes_Arrival_to_Departure_by_period.png', 'graph_mean_Minutes_Arrival_to_Enga

## 2. Helper Function for Cleaning Column Names

This function standardizes column names so they are easier to use in Python.

For example, a column like:

`Dispatch Request Date & Time`

becomes:

`dispatch_request_date_time`

In [63]:
def clean_column_names(df):
    """
    Standardizes column names by making them lowercase,
    replacing spaces/symbols with underscores, and removing extra underscores.
    """
    df = df.copy()
    
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace("&", "and", regex=False)
        .str.replace("#", "number", regex=False)
        .str.replace(":", "", regex=False)
        .str.replace("→", "to", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace(" ", "_", regex=False)
        .str.replace("__", "_", regex=False)
    )
    
    return df

## 3. Load Raw CSV Files

This function loads the raw CSV files needed for the project.

The four main datasets are:

- MCSLC data
- SPD calls
- SPD close codes
- SPD units

In [64]:
def load_csv_files(data_folder=DATA_FOLDER):
    """
    Loads the raw project CSV files from the same folder as the notebook.
    """
    mcslc = pd.read_csv(data_folder / "MCSLC.csv")
    spd_calls = pd.read_csv(data_folder / "2015-2025 SPD Calls for Service.csv")
    spd_close_codes = pd.read_csv(data_folder / "2015-2025 SPD Calls with Close Codes.csv")
    spd_units = pd.read_csv(data_folder / "2015-2025 SPD Responding Units.csv")
    
    return mcslc, spd_calls, spd_close_codes, spd_units

## 4. Clean MCSLC Data

This function cleans the main MCSLC dataset.

Main cleaning steps:

- Standardize column names
- Convert date/time columns to datetime
- Convert response-time columns to numeric
- Create a rollout period variable
- Create a month variable
- Create a year variable

In [65]:
def clean_mcslc_data(mcslc):
    """
    Cleans the MCSLC dataset for analysis.
    """
    mcslc = clean_column_names(mcslc)
    
    # Convert MCSLC date/time columns using the exact format from the data
    datetime_cols = [
        "dispatch_request_date_and_time",
        "dispatch_date_and_time",
        "arrival_on_scene_date_and_time",
        "engagement_with_client_date_and_time",
        "mcit_departure_date_and_time"
    ]
    
    for col in datetime_cols:
        if col in mcslc.columns:
            mcslc[col] = pd.to_datetime(
                mcslc[col],
                format="%m/%d/%y %H:%M",
                errors="coerce"
            )
    
    # Convert response time columns to numeric
    response_time_cols = [
        "minutes_request_to_dispatch",
        "minutes_dispatch_to_arrival",
        "minutes_arrival_to_engagement",
        "minutes_arrival_to_departure"
    ]
    
    for col in response_time_cols:
        if col in mcslc.columns:
            mcslc[col] = pd.to_numeric(mcslc[col], errors="coerce")
    
    # MCSLC rollout date
    rollout_date = pd.Timestamp("2024-08-01")
    
    # Create before/after rollout variable
    if "dispatch_request_date_and_time" in mcslc.columns:
        mcslc["rollout_period"] = np.where(
            mcslc["dispatch_request_date_and_time"] < rollout_date,
            "Before Rollout",
            "After Rollout"
        )
        
        mcslc["dispatch_month"] = mcslc["dispatch_request_date_and_time"].dt.to_period("M").astype(str)
        mcslc["dispatch_year"] = mcslc["dispatch_request_date_and_time"].dt.year
    
    return mcslc

## 5. Create Long-Format Dispatch Reasons Dataset

The original MCSLC data has several reason columns:

- Reason for Dispatch #1
- Reason for Dispatch #2
- Reason for Dispatch #3
- Reason for Dispatch #4
- Reason for Dispatch #5

This function reshapes those columns into one long-format dataset. This makes it easier to count and graph dispatch reasons.

In [66]:
def create_reasons_long(mcslc_clean):
    """
    Creates a long-format dataset for dispatch reasons.
    """
    reason_cols = [
        col for col in mcslc_clean.columns
        if "reason_for_dispatch" in col
    ]
    
    id_cols = [
        col for col in [
            "id",
            "end_point_of_dispatch",
            "city",
            "dispatch_request_date_and_time",
            "rollout_period",
            "dispatch_month",
            "dispatch_year"
        ]
        if col in mcslc_clean.columns
    ]
    
    reasons_long = mcslc_clean[id_cols + reason_cols].melt(
        id_vars=id_cols,
        value_vars=reason_cols,
        var_name="reason_number",
        value_name="reason_for_dispatch"
    )
    
    # Remove blank reasons
    reasons_long = reasons_long.dropna(subset=["reason_for_dispatch"])
    reasons_long = reasons_long[
        reasons_long["reason_for_dispatch"].astype(str).str.strip() != ""
    ]
    
    return reasons_long

## 6. Clean SPD Calls Data

This function cleans the SPD calls dataset.

The exact columns may vary depending on the export, so the function is written to safely clean the file without breaking if certain columns are missing.

In [67]:
def clean_spd_calls_data(spd_calls):
    """
    Cleans the SPD calls dataset.
    """
    spd_calls = clean_column_names(spd_calls)
    
    # SPD columns are time-only, not full dates
    time_cols = [
        "call_creation_time",
        "first_dispatched_time",
        "first_arrival_time",
        "clear_time"
    ]
    
    for col in time_cols:
        if col in spd_calls.columns:
            spd_calls[col] = pd.to_datetime(
                spd_calls[col],
                format="%I:%M %p",
                errors="coerce"
            ).dt.time
    
    return spd_calls

## 7. Filter SPD Calls to Relevant Crisis-Response Calls

This section creates a smaller SPD dataset focused on calls that may be relevant to the project.

The keywords can be adjusted depending on the exact call categories in the data.

In [68]:
def filter_spd_relevant_calls(spd_calls_clean):
    """
    Filters SPD calls to calls that are likely relevant to behavioral health,
    welfare checks, crisis response, medical calls, or related public safety calls.
    """
    keywords = [
        "mental",
        "welfare",
        "suicide",
        "crisis",
        "disorderly",
        "medical",
        "assist",
        "subject",
        "behavior",
        "intox",
        "overdose"
    ]
    
    # Select text-like columns without using select_dtypes(include="object")
    text_cols = [
        col for col in spd_calls_clean.columns
        if spd_calls_clean[col].dtype == "object" or pd.api.types.is_string_dtype(spd_calls_clean[col])
    ]
    
    combined_text = (
        spd_calls_clean[text_cols]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
        .str.lower()
    )
    
    keyword_pattern = "|".join(keywords)
    spd_relevant = spd_calls_clean[combined_text.str.contains(keyword_pattern, na=False)].copy()
    
    return spd_relevant

## 8. Clean SPD Close Codes Data

This function standardizes the SPD close codes dataset.

In [69]:
def clean_spd_units_data(spd_units):
    """
    Cleans the SPD units dataset.
    """
    spd_units = clean_column_names(spd_units)
    return spd_units

## 9. Clean SPD Units Data

This function standardizes the SPD units dataset.

It also tries to convert date/time columns if they are present.

In [70]:
def clean_spd_units_data(spd_units):
    """
    Cleans the SPD units dataset.
    """
    spd_units = clean_column_names(spd_units)
    
    datetime_cols = [
        col for col in spd_units.columns
        if "date" in col or "time" in col
    ]
    
    for col in datetime_cols:
        spd_units[col] = pd.to_datetime(spd_units[col], errors="coerce")
    
    return spd_units

## 10. Full Cleaning Function

This function runs the full cleaning pipeline.

It loads all raw files, cleans them, creates the long-format reasons dataset, filters relevant SPD calls, and saves cleaned CSV files.

In [71]:
def updated_cleaned_data(data_folder=DATA_FOLDER, save_files=True):
    """
    Runs the full cleaning pipeline.

    Returns:
    - mcslc_clean
    - reasons_long_clean
    - spd_calls_clean
    - spd_relevant
    - spd_close_codes_clean
    - spd_units_clean
    """
    
    # Load raw CSV files
    mcslc, spd_calls, spd_close_codes, spd_units = load_csv_files(data_folder=data_folder)
    
    # Clean each dataset
    mcslc_clean = clean_mcslc_data(mcslc)
    reasons_long_clean = create_reasons_long(mcslc_clean)
    spd_calls_clean = clean_spd_calls_data(spd_calls)
    spd_relevant = filter_spd_relevant_calls(spd_calls_clean)
    spd_close_codes_clean = clean_spd_close_codes_data(spd_close_codes)
    spd_units_clean = clean_spd_units_data(spd_units)
    
    # Save cleaned files
    if save_files:
        mcslc_clean.to_csv(data_folder / "mcslc_clean.csv", index=False)
        reasons_long_clean.to_csv(data_folder / "reasons_long_clean.csv", index=False)
        spd_calls_clean.to_csv(data_folder / "spd_calls_clean.csv", index=False)
        spd_relevant.to_csv(data_folder / "spd_relevant_calls_clean.csv", index=False)
        spd_close_codes_clean.to_csv(data_folder / "spd_close_codes_clean.csv", index=False)
        spd_units_clean.to_csv(data_folder / "spd_units_clean.csv", index=False)
    
    return (
        mcslc_clean,
        reasons_long_clean,
        spd_calls_clean,
        spd_relevant,
        spd_close_codes_clean,
        spd_units_clean
    )

## 11. Run Cleaning Pipeline

This cell runs the full cleaning function and prints quick checks to make sure the cleaned files were created correctly.

In [72]:
if __name__ == "__main__":
    (
        mcslc_clean,
        reasons_long_clean,
        spd_calls_clean,
        spd_relevant,
        spd_close_codes_clean,
        spd_units_clean
    ) = updated_cleaned_data()

    print("\nQuick checks after cleaning:")
    print("MCSLC cleaned shape:", mcslc_clean.shape)
    print("Reasons long shape:", reasons_long_clean.shape)
    print("SPD calls cleaned shape:", spd_calls_clean.shape)
    print("SPD relevant calls shape:", spd_relevant.shape)
    print("SPD close codes cleaned shape:", spd_close_codes_clean.shape)
    print("SPD units cleaned shape:", spd_units_clean.shape)


Quick checks after cleaning:
MCSLC cleaned shape: (4682, 21)
Reasons long shape: (9185, 9)
SPD calls cleaned shape: (54804, 11)
SPD relevant calls shape: (7288, 11)
SPD close codes cleaned shape: (98, 2)
SPD units cleaned shape: (56, 6)


## 12. Preview Cleaned Data

This final section previews the main cleaned datasets.

In [73]:
mcslc_clean.head()

,id,end_point_of_dispatch,city,dispatch_request_date_and_time,dispatch_date_and_time,arrival_on_scene_date_and_time,engagement_with_client_date_and_time,mcit_departure_date_and_time,reason_for_dispatch_number1,reason_for_dispatch_number2,...,reason_for_dispatch_number4,reason_for_dispatch_number5,disposition,minutes_request_to_dispatch,minutes_dispatch_to_arrival,minutes_arrival_to_engagement,minutes_arrival_to_departure,rollout_period,dispatch_month,dispatch_year
0,12236,Cancelled,Eugene,2025-05-12 13:29:00,2025-05-12 13:34:00,2025-05-12 00:00:00,2025-05-12 00:00:00,2025-05-12 14:03:00,Harm/risk of harm to self/others/property,Disorganized behavior,...,NaN,NaN,Arrest,5.0,NaN,0.0,NaN,After Rollout,2025-05,2025.0
1,12574,Cancelled,Unknown,2025-12-05 17:41:00,2025-12-05 17:47:00,2025-12-05 18:06:00,NaT,2025-12-05 18:12:00,Agitation or disruptive behavior,NaN,...,NaN,NaN,Arrest,6.0,19.0,NaN,NaN,After Rollout,2025-12,2025.0
2,12888,Cancelled,Eugene,2025-07-25 21:10:00,2025-07-25 21:14:00,2025-07-25 21:28:00,2025-07-25 21:30:00,2025-07-25 21:36:00,Agitation or disruptive behavior,NaN,...,NaN,NaN,Arrest,4.0,14.0,2.0,8.0,After Rollout,2025-07,2025.0
3,10504,Engaged Client,Eugene,2025-03-29 22:29:00,2025-03-29 22:43:00,2025-03-29 23:21:00,2025-03-29 23:28:00,2025-03-29 23:58:00,Harm/risk of harm to self/others/property,Disorganized behavior,...,NaN,NaN,Arrest,14.0,38.0,7.0,37.0,After Rollout,2025-03,2025.0
4,12220,Engaged Client,Eugene,2025-04-26 18:52:00,2025-04-26 18:54:00,2025-04-26 19:05:00,2025-04-26 19:07:00,2025-04-26 19:32:00,Disorganized behavior,Agitation or disruptive behavior,...,NaN,NaN,Arrest,2.0,11.0,2.0,27.0,After Rollout,2025-04,2025.0


In [74]:
reasons_long_clean.head()

,id,end_point_of_dispatch,city,dispatch_request_date_and_time,rollout_period,dispatch_month,dispatch_year,reason_number,reason_for_dispatch
0,12236,Cancelled,Eugene,2025-05-12 13:29:00,After Rollout,2025-05,2025.0,reason_for_dispatch_number1,Harm/risk of harm to self/others/property
1,12574,Cancelled,Unknown,2025-12-05 17:41:00,After Rollout,2025-12,2025.0,reason_for_dispatch_number1,Agitation or disruptive behavior
2,12888,Cancelled,Eugene,2025-07-25 21:10:00,After Rollout,2025-07,2025.0,reason_for_dispatch_number1,Agitation or disruptive behavior
3,10504,Engaged Client,Eugene,2025-03-29 22:29:00,After Rollout,2025-03,2025.0,reason_for_dispatch_number1,Harm/risk of harm to self/others/property
4,12220,Engaged Client,Eugene,2025-04-26 18:52:00,After Rollout,2025-04,2025.0,reason_for_dispatch_number1,Disorganized behavior


In [75]:
spd_relevant.head()

,incident_number,initial_call_type,final_call_type,responding_agency,primary_responding_unit,call_creation_time,first_dispatched_time,first_arrival_time,clear_time,priority,call_creation_mechanism
67,15003714,SUSPSU,SUSPICIOUS SUBJECT,SPD,1S22,NaT,NaT,NaT,NaT,3,PHONE
86,15004415,SUSPSU,SUSPICIOUS SUBJECT,SPD,3S22,NaT,NaT,NaT,NaT,4,PHONE
93,15004706,SUSPSU,SUSPICIOUS SUBJECT,SPD,1S22,NaT,NaT,NaT,NaT,3,E911
109,15005301,SUBSCR,SUBJECT SCREAMING,SPD,3S13,NaT,NaT,NaT,NaT,3,PHONE
116,15005988,SUSPSU,SUSPICIOUS SUBJECT,SPD,3S12,NaT,NaT,NaT,NaT,4,PHONE
